<a href="https://colab.research.google.com/github/lokesh2004-g/NLP-Project/blob/main/simple_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd


In [3]:
df=pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [42]:
#tokenize
def tokenize(text):
  text=text.lower()
  text=text.replace('?','')
  text=text.replace("'","")
  return text.split()

In [43]:
tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [44]:
#vocablary
vocab={'<UNK>':0}

In [45]:
def build_vocab(row):
  tokenize_questions=tokenize(row['question'])
  tokenize_answer=tokenize(row['answer'])
  merge_tokens=tokenize_answer+tokenize_questions
  for token in merge_tokens:
    if token not in vocab:
      vocab[token]=len(vocab)


In [46]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [47]:
vocab

{'<UNK>': 0,
 'paris': 1,
 'what': 2,
 'is': 3,
 'the': 4,
 'capital': 5,
 'of': 6,
 'france': 7,
 'berlin': 8,
 'germany': 9,
 'harper-lee': 10,
 'who': 11,
 'wrote': 12,
 'to': 13,
 'kill': 14,
 'a': 15,
 'mockingbird': 16,
 'jupiter': 17,
 'largest': 18,
 'planet': 19,
 'in': 20,
 'our': 21,
 'solar': 22,
 'system': 23,
 '100': 24,
 'boiling': 25,
 'point': 26,
 'water': 27,
 'celsius': 28,
 'leonardo-da-vinci': 29,
 'painted': 30,
 'mona': 31,
 'lisa': 32,
 '8': 33,
 'square': 34,
 'root': 35,
 '64': 36,
 'au': 37,
 'chemical': 38,
 'symbol': 39,
 'for': 40,
 'gold': 41,
 '1945': 42,
 'which': 43,
 'year': 44,
 'did': 45,
 'world': 46,
 'war': 47,
 'ii': 48,
 'end': 49,
 'nile': 50,
 'longest': 51,
 'river': 52,
 'tokyo': 53,
 'japan': 54,
 'albert-einstein': 55,
 'developed': 56,
 'theory': 57,
 'relativity': 58,
 '32': 59,
 'freezing': 60,
 'fahrenheit': 61,
 'mars': 62,
 'known': 63,
 'as': 64,
 'red': 65,
 'george-orwell': 66,
 'author': 67,
 '1984': 68,
 'pound': 69,
 'currenc

In [48]:
len(vocab)

324

In [49]:
def text_to_indices(text,vocab):
  index_text=[]
  for token in tokenize(text):
    if token in vocab:
      index_text.append(vocab[token])
    else:
      index_text.append(vocab['<UNK>'])


  return index_text

In [50]:
text_to_indices("What is my name",vocab)

[2, 3, 0, 0]

In [55]:
import torch
from torch.utils.data import  Dataset ,DataLoader

In [65]:
class QansDataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question=text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer=text_to_indices(self.df.iloc[index]['answer'],self.vocab)
    return torch.tensor(numerical_question),torch.tensor(numerical_answer)



In [66]:
dataset=QansDataset(df,vocab)

In [70]:
dataset[0]

(tensor([2, 3, 4, 5, 6, 7]), tensor([1]))

In [72]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [73]:
for question,answer in dataloader:
  print(question,answer)

tensor([[ 11, 141,   4, 142, 143,  13, 144,  84,   4, 145]]) tensor([[140]])
tensor([[ 11,  12, 190, 159, 191]]) tensor([[189]])
tensor([[ 11,   3,  63,  64,   4, 284,   6, 285]]) tensor([[283]])
tensor([[  2,   3,   4,  18, 116,  84,  85]]) tensor([[115]])
tensor([[ 43, 291, 292, 119, 293, 159, 294, 295]]) tensor([[290]])
tensor([[ 43, 217, 119, 218, 219,  20,  15, 220,  44]]) tensor([[216]])
tensor([[  2,   3,   4,   5,   6, 114]]) tensor([[113]])
tensor([[2, 3, 4, 5, 6, 7]]) tensor([[1]])
tensor([[ 43, 118, 119,   4, 120,  95, 121]]) tensor([[117]])
tensor([[ 11,  76, 112]]) tensor([[111]])
tensor([[ 43, 102,   3,   4,  18]]) tensor([[101]])
tensor([[  2,   3,   4, 147,  87,  20, 193, 194]]) tensor([[192]])
tensor([[ 43,  87,  88, 242, 243,  20,  40, 244]]) tensor([[241]])
tensor([[  2,   3,   4,  93, 137,  20,   4,  46]]) tensor([[185]])
tensor([[ 43, 168,   3,   4,  18, 169, 170]]) tensor([[167]])
tensor([[ 11,  76, 209]]) tensor([[208]])
tensor([[11, 76, 77]]) tensor([[75]])
tens

In [74]:
import torch.nn as nn

In [89]:
class SimpleRnn(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True)
    self.outlayer=nn.Linear(64,vocab_size)
  def forward(self,question):
    emeb_question=self.embedding(question)
    hidden,final=self.rnn(emeb_question)
    output=self.outlayer(final.squeeze(0))
    return output


In [90]:
learning_rate=0.001
epochs=20

In [91]:
model=SimpleRnn(len(vocab))

In [92]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [93]:
for epoch in range(epochs):
  total_loss=0
  for question, answer in dataloader:
    optimizer.zero_grad()
    output=model(question)
    loss=criterion(output,answer[0])
    loss.backward()
    optimizer.step()
    total_loss=total_loss+loss.item()
  print(f'loss:{total_loss} epoch {epoch}')

loss:526.3536348342896 epoch 0
loss:453.89380502700806 epoch 1
loss:373.14957189559937 epoch 2
loss:314.0821268558502 epoch 3
loss:263.3827438354492 epoch 4
loss:215.24214923381805 epoch 5
loss:172.33228731155396 epoch 6
loss:134.91136574745178 epoch 7
loss:104.63899332284927 epoch 8
loss:79.89688494801521 epoch 9
loss:61.99301120638847 epoch 10
loss:48.57312847673893 epoch 11
loss:38.80313827097416 epoch 12
loss:31.41982391476631 epoch 13
loss:25.7486906722188 epoch 14
loss:21.61268512904644 epoch 15
loss:18.13682059943676 epoch 16
loss:15.276350155472755 epoch 17
loss:13.187417447566986 epoch 18
loss:11.38812755420804 epoch 19


In [101]:
def predict(model,question,threshold=.3):
  numerical_question=text_to_indices(question,vocab)
  question_tensor=torch.tensor(numerical_question).unsqueeze(0)
  model.eval()
  output=model(question_tensor)
  probs=torch.nn.functional.softmax(output,dim=1)
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])



In [102]:
predict(model, "What is the largest planet in our solar system?")

jupiter
